In [33]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEndpointEmbeddings
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
from langchain_core.runnables import RunnableLambda,RunnableParallel,RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [2]:
loader=PyPDFLoader("Anthropogenic_disturbance.pdf")
data=loader.load()


In [3]:
print(data[0].page_content)

Anthropogenic disturbance expands the climatic
limits of annual plant dominance
July 10, 2026
Keywords:climate change; functional groups; functional traits; grasslands; life cy-
cle; life-history; lifespan; perennials; rainfall; Disturbance and Recovery Across Global
Grasslands Network (DRAGNet)
Abstract
Disturbance regimes and nutrient inputs are changing worldwide, with con-
sequences for the structure and functioning of plant communities. Classical life-
history theory predicts that disturbance should shift communities from long-lived
perennials toward short-lived annuals, and that nutrient enrichment may amplify
this shift. However, these predictions have not been tested experimentally across
broad environmental gradients. Here, using a global coordinated grassland exper-
iment spanning 37 sites, we tested how physical disturbance (vegetation removal
and shallow soil tillage) and fertilisation reshape annual–perennial balance, and
whether disturbance relaxes the climatic limits of 

In [4]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
docs=splitter.split_documents(data)

In [5]:
print(docs[0])

page_content='Anthropogenic disturbance expands the climatic
limits of annual plant dominance
July 10, 2026
Keywords:climate change; functional groups; functional traits; grasslands; life cy-
cle; life-history; lifespan; perennials; rainfall; Disturbance and Recovery Across Global
Grasslands Network (DRAGNet)
Abstract
Disturbance regimes and nutrient inputs are changing worldwide, with con-
sequences for the structure and functioning of plant communities. Classical life-
history theory predicts that disturbance should shift communities from long-lived
perennials toward short-lived annuals, and that nutrient enrichment may amplify
this shift. However, these predictions have not been tested experimentally across
broad environmental gradients. Here, using a global coordinated grassland exper-
iment spanning 37 sites, we tested how physical disturbance (vegetation removal
and shallow soil tillage) and fertilisation reshape annual–perennial balance, and' metadata={'producer': 'pikepdf 8.15.

In [6]:
embedding = HuggingFaceEndpointEmbeddings(model='sentence-transformers/all-MiniLM-L6-v2',
                                           task='feature-extraction')

In [7]:
vectorstore = Chroma(
    collection_name="anthropogenic_disturbance",
    embedding_function=embedding,
    persist_directory="./chroma_db"
)

In [8]:
vectorstore.add_documents(docs)

['d8c57a93-990a-41ce-94e5-12b3d0c3f7f1',
 '039988dc-7f4b-4cb5-8e48-0add5af6c7ac',
 'ccbd09d4-05b4-4cac-b18b-ffe4013c88f2',
 'd3a7cbe3-cbb3-45c9-b160-4658f3d6c2a9',
 'b5e343d1-0aa7-4828-b49d-e8496d2141f9',
 'b9d8f38b-aded-42e5-b249-8b9691a70392',
 '9678a9de-8749-409a-bd7c-e0f0bbdfc81a',
 'd84eab5f-1ae3-4351-873d-47e15df3d6cf',
 '3a379904-e1a0-4c18-a060-2a02a5608fae',
 '3bd65776-0dea-4711-800f-8921a4046fe8',
 'd03792ef-5b3c-481e-a435-b5b366f0edd0',
 'c7df9d6c-c24c-43d7-9595-3213eeff8515',
 'fce59b60-3269-4339-a7fe-e259569f90ed',
 '994fa4a9-4eb4-44eb-b77c-86d08f2f854c',
 'a1f232aa-0900-4f30-864a-cffe29c6ded9',
 '0f38faec-614a-4de2-95ab-72d9fd2a1751',
 '26995901-02cd-4d72-b81b-e3aa6ebe28e2',
 'dc61d012-f488-49f1-bcec-a172986d2f7f',
 'bee831b9-7b2d-4a49-aed8-63cf92b45fee',
 'a2eaa196-e5f9-4e01-aa6f-77ce3267af2e',
 '54bd410b-129f-4513-baf6-7c92f7c9bbe4',
 '302ed005-4835-4e4a-9b31-45ff70655a73',
 '472315e4-5cd7-4bd2-a57c-cc545e9fc56c',
 'e796f757-4e27-4e4d-843a-3e724b6056aa',
 'b672bd57-01f9-

In [70]:
def input_question(x):
    question=(x)
    return question

input_question_runnable = RunnableLambda(input_question)


In [20]:
def format_docs(retrived_docs):
    context_text="\n\n".join([doc.page_content for doc in retrived_docs])
    return context_text

format_docs_runnable = RunnableLambda(format_docs)

In [13]:
# prompt_question=PromptTemplate(
#     template="""
#     {question}""",

#     inputvariables=["question"]

# )
prompt_final=PromptTemplate(
    template= "You are a helpful assistant. Use the following context to answer the question at the end. If you don't know the answer, just say you don't know, don't try to make up an answer. Here is the context:  {context} Now answer the question with the help of the context provided. If the context is not sufficient to answer the question, say I don't know. Question: {question}"
    ,
    inputvariables=["context","question"]
)

In [10]:
load_dotenv()

True

In [14]:
llm=HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
)
model = ChatHuggingFace(
llm=llm,
)

In [15]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k":4, 
                   "fetch_k":10,}
)

In [47]:
parser=StrOutputParser()

In [16]:
passchain=RunnablePassthrough(
    question=RunnablePassthrough(),
)

In [22]:
# chain = input_question_runnable | prompt_question | retriever | prompt_final | model

parallel_chain = RunnableParallel({
    'context': retriever | format_docs_runnable ,
    'question': RunnablePassthrough()}
)
final_chain=  parallel_chain | prompt_final | model


In [24]:
result=final_chain.invoke("what is Land-use intensification?")
print(result)

content='Land-use intensification, as discussed in the reference [2] by Gossner et al., refers to the practice of increasing the intensity of land management activities, which can lead to the homogenization of grassland communities across different trophic levels. This process involves changes in land management practices that can alter the structure and function of ecosystems.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 71, 'prompt_tokens': 817, 'total_tokens': 888}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019f6f2e-92f8-7172-ad39-247160b365f2-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 817, 'output_tokens': 71, 'total_tokens': 888}


In [30]:
result=final_chain.invoke("Annual plants are often viewed as transient or subordinate components of grassland ecosystems. But What did  Our results instead show?")
print(result)

content='Our results instead show that disturbance can shift the annual–perennial balance consistently across grasslands worldwide and expand the climatic conditions under which annual strategies dominate.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 738, 'total_tokens': 770}, 'model_name': 'Qwen/Qwen2.5-7B-Instruct', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019f6f34-ed9c-7810-a4ca-6038b8c5de16-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 738, 'output_tokens': 32, 'total_tokens': 770}


In [29]:
what_was_retrived=format_docs(retriever.invoke("what is Land-use intensification?"))

print(what_was_retrived)

References
[1] Foley JA, DeFries R, Asner GP, Barford C, Bonan G, Carpenter SR, et al. Global
consequences of land use. Science. 2005;309:570-4.
[2] Gossner MM, Lewinsohn TM, Kahl T, Grassein F, Boch S, Prati D, et al. Land-
use intensification causes multitrophic homogenization of grassland communities.
Nature. 2016 Nov;540(7632):266-9. Available from:http://www.nature.com/
doifinder/10.1038/nature20575.
[3] Friedman J, Rubin MJ. All in good time: Understanding annual and perennial
strategies in plants. American Journal of Botany. 2015;102(4):497-9.
[4] Vico G, Manzoni S, Nkurunziza L, Murphy K, Weih M. Trade-offs between seed
output and life span - a quantitative comparison of traits between annual and
perennial congeneric species. New Phytologist. 2016;209(1):104-14.
[5] Glover JD, Reganold JP, Bell L W, Borevitz J, Brummer EC, Buckler ES,
et al. Increased Food and Ecosystem Security via Perennial Grains. Science.
2010;328(5986):1638-9.

year for three consecutive years and is desig

In [31]:
what_was_retrived=format_docs(retriever.invoke("Annual plants are often viewed as transient or subordinate components of grassland ecosystems. But What did  Our results instead show?"))

print(what_was_retrived)

1 Introduction
Land-use intensification is transforming grasslands worldwide through increased distur-
bances and nutrient inputs, with consequences for vegetation structure and ecosystem
functioning 1;2. These changes may reorganize grassland communities by shifting the
balance between annual and perennial life-history strategies. Annuals represent the
short-lived extreme of plant life span, completing their life cycle within a year, while
perennials persist for longer and often reproduce repeatedly 3. Hence, annuals often
allocate more to rapid aboveground growth and seed production, while perennials tend
to invest more in storage organs and belowground structures 4. Because perennials gen-
erally contribute more to long-term carbon storage, soil stabilisation, and hydrological
regulation 4;5, shifts in annual–perennial balance can have important consequences for
ecosystem functioning.
Annual species constitute∼15% of the world’s herbaceous flora, with substantially

3.4 Conclusions
